# Vježba: hoće li se mašina pokvariti?

Isti postupak kao s korisnicima telekom operatera, samo na drugim podacima: **senzori na CNC mašini** (AI4I 2020, UCI).
10.000 mjerenja, a za svako znamo temperature, brzinu okretaja, moment, habanje alata i da li je došlo do kvara.

Ćelije s `# TODO` popunjavate vi. Gdje piše `___`, tu nešto nedostaje.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

df = pd.read_csv("ai4i2020.csv")
df.head()

## 1. Koliko često se mašina pokvari?

In [ ]:
# TODO: procenat kvarova
print(f"Kvarova: {df.kvar.___() * 100:.1f} %")

## 2. Train, validacija, test (60 / 20 / 20)

`stratify` čuva isti procenat kvarova u svakom skupu, jer ih je malo.
Kolone `kvar`, `vrsta_kvara` i `id` ne idu u ulaze. Zašto `vrsta_kvara`?

In [ ]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1, stratify=df.kvar)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1, stratify=df_full_train.kvar)

y_train = df_train.kvar.values
y_val = df_val.kvar.values

kolone = ["tip", "temp_zraka_C", "temp_procesa_C", "brzina_okretaja_rpm", "moment_Nm", "habanje_alata_min"]
len(df_train), len(df_val), len(df_test)

## 3. One-hot encoding (kolona `tip`: L, M, H)

In [ ]:
# TODO: fit_transform na train, samo transform na validaciji
dv = DictVectorizer(sparse=False)
X_train = dv.___(df_train[kolone].to_dict(orient="records"))
X_val = dv.___(df_val[kolone].to_dict(orient="records"))
dv.get_feature_names_out()

## 4. Logistička regresija i evaluacija

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)
y_pred = model.predict_proba(X_val)[:, 1]
odluka = (y_pred >= 0.5)

print(f"Accuracy: {accuracy_score(y_val, odluka) * 100:.1f} %")
print(f"Model 'nikad kvar': {(y_val == 0).mean() * 100:.1f} %")

In [ ]:
# TODO: matrica konfuzije, precision i recall
tp = (odluka & (y_val == 1)).sum()
fp = (odluka & (y_val == 0)).sum()
fn = (~odluka & (y_val == 1)).sum()
tn = (~odluka & (y_val == 0)).sum()
print("TN", tn, " FP", fp, " FN", fn, " TP", tp)
print(f"Precision: {tp / (___):.2f}   Recall: {tp / (___):.2f}")

**Pitanje:** accuracy je oko 97 %. Je li ovo dobar model? Koliko kvarova od 68 je uhvatio?

## 5. Probajte druge klasifikatore

Svi imaju isti `fit` i `predict`. Poredimo ih po **F1** (spaja precision i recall).
- **Stablo odlučivanja:** postavlja pitanja da/ne (je li moment > 60?)
- **Random forest:** 100 stabala glasa
- **KNN:** gleda 5 najsličnijih mjerenja iz train skupa
- **Gradient boosting:** stabla se grade jedno za drugim, svako popravlja greške prethodnog

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

modeli = {
    "logistička regresija": LogisticRegression(max_iter=1000),
    "stablo odlučivanja": DecisionTreeClassifier(max_depth=5, random_state=1),
    "random forest": RandomForestClassifier(n_estimators=100, random_state=1),
    "KNN": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "gradient boosting": GradientBoostingClassifier(random_state=1),
}

def uporedi(X_tr, X_v):
    for naziv, m in modeli.items():
        m.fit(X_tr, y_train)
        p = m.predict(X_v)
        print(f"{naziv:22s} accuracy {accuracy_score(y_val, p):.3f}   precision {precision_score(y_val, p, zero_division=0):.2f}   "
              f"recall {recall_score(y_val, p):.2f}   F1 {f1_score(y_val, p):.2f}   uhvaćeno {((p == 1) & (y_val == 1)).sum()}/68")

uporedi(X_train, X_val)

**Pitanje:** koji model je najbolji po F1? A po accuracy? Je li to isti model?

## 6. Nove kolone iz fizike

Snaga P = M · ω (ω = 2π · n / 60), razlika temperatura, i habanje × moment.

In [ ]:
# TODO: napravite nove kolone
for d in [df_train, df_val]:
    d["snaga_W"] = d.moment_Nm * d.brzina_okretaja_rpm * 2 * np.pi / 60
    d["razlika_temp_C"] = ___ - ___
    d["habanje_x_moment"] = ___

kolone_nove = kolone + ["snaga_W", "razlika_temp_C", "habanje_x_moment"]
dv = DictVectorizer(sparse=False)
X_train2 = dv.fit_transform(df_train[kolone_nove].to_dict(orient="records"))
X_val2 = dv.transform(df_val[kolone_nove].to_dict(orient="records"))

uporedi(X_train2, X_val2)

**Pitanje:** šta je više pomoglo, drugi model ili nove kolone?